In [0]:
%sql
DROP TABLE IF EXISTS payment_app.gold.dim_biller

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as F

# =====================================================
# nb_dim_biller (Simplified SCD Type 2)
# No created_timestamp / updated_timestamp
# =====================================================

gold_table = "payment_app.gold.dim_biller"

# =====================================================
# Read Current Snapshot from Silver
# =====================================================
src = (
    spark.table("payment_app.silver.billers")
    .filter(F.col("batch_id") == v_batch_id)
    .select(
        "biller_id",
        "biller_name",
        "biller_category",
        "status"
    )
    .dropDuplicates(["biller_id"])
)

# =====================================================
# First Load
# =====================================================
if not spark.catalog.tableExists(gold_table):

    first_load = (
        src
        .withColumn("biller_key", monotonically_increasing_id() + 1)
        .withColumn("effective_from", to_date(lit(v_batch_id)))
        .withColumn("effective_to", lit(None).cast("date"))
        .withColumn("is_current", lit("Y"))
        .select(
            "biller_key",
            "biller_id",
            "biller_name",
            "biller_category",
            "status",
            "effective_from",
            "effective_to",
            "is_current"
        )
    )

    first_load.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(gold_table)

# =====================================================
# Incremental Load
# =====================================================
else:

    current_dim = (
        spark.table(gold_table)
        .filter("is_current = 'Y'")
    )

    # -------------------------------------------------
    # Find New or Changed Billers
    # -------------------------------------------------
    changes = (
        src.alias("s")
        .join(current_dim.alias("t"), "biller_id", "left")
        .filter(
            col("t.biller_id").isNull() |
            (coalesce(col("s.biller_name"), lit("")) != coalesce(col("t.biller_name"), lit(""))) |
            (coalesce(col("s.biller_category"), lit("")) != coalesce(col("t.biller_category"), lit(""))) |
            (coalesce(col("s.status"), lit("")) != coalesce(col("t.status"), lit("")))
        )
        .select("s.*")
    )

    changes.createOrReplaceTempView("vw_changes")

    # -------------------------------------------------
    # Expire Existing Current Rows
    # -------------------------------------------------
    spark.sql(f"""
        MERGE INTO {gold_table} tgt
        USING vw_changes src
        ON tgt.biller_id = src.biller_id
        AND tgt.is_current = 'Y'

        WHEN MATCHED THEN UPDATE SET
            tgt.effective_to = date_sub(to_date('{v_batch_id}'), 1),
            tgt.is_current = 'N'
    """)

    # -------------------------------------------------
    # Get Next Surrogate Key
    # -------------------------------------------------
    max_key = spark.sql(f"""
        SELECT COALESCE(MAX(biller_key),0) AS max_key
        FROM {gold_table}
    """).collect()[0]["max_key"]

    # -------------------------------------------------
    # Insert New Versions
    # -------------------------------------------------
    final_insert = (
        changes
        .withColumn("biller_key", monotonically_increasing_id() + max_key + 1)
        .withColumn("effective_from", to_date(lit(v_batch_id)))
        .withColumn("effective_to", lit(None).cast("date"))
        .withColumn("is_current", lit("Y"))
        .select(
            "biller_key",
            "biller_id",
            "biller_name",
            "biller_category",
            "status",
            "effective_from",
            "effective_to",
            "is_current"
        )
    )

    final_insert.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(gold_table)